In [ ]:
# ==========================================
# 0. 라이브러리 임포트 및 한글 폰트 설정
# 0. 导入库及设置韩文字体
# 0. Import Libraries and Set Korean Font
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

# 콜랩에서 한글 그래프 깨짐 방지 (나눔고딕 폰트 설치 및 설정)
# 防止Colab中韩文图表乱码（安装并设置NanumGothic字体）
# Prevent Korean graph corruption in Colab (install and set NanumGothic font)
!apt-get install -y fonts-nanum
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지 / 防止负号乱码 / Prevent minus sign corruption

# ==========================================
# 1. 데이터 업로드 (사용자가 직접 파일 선택)
# 1. 数据上传（用户直接选择文件）
# 1. Data Upload (User selects file directly)
# ==========================================
print("👇 아래 '파일 선택' 버튼이 나타나면 'payments(1).csv' 파일을 선택하여 업로드하세요.")
uploaded = files.upload()

# 업로드된 파일의 이름을 자동으로 가져옵니다.
# 自动获取上传文件的名称。
# Automatically retrieve the name of the uploaded file.
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f"\n✅ '{filename}' 파일 로드 완료! 데이터 크기: {df.shape}")

# ==========================================
# 2. 변수 분리 (Target y, Input X) 및 전처리
# 2. 变量分离（目标变量y，输入变量X）及预处理
# 2. Variable Separation (Target y, Input X) and Preprocessing
# ==========================================
# Target 변수(y)와 Input 변수(X) 분리
# 分离目标变量(y)和输入变量(X)
# Separate Target variable (y) and Input variable (X)
y = df['is_new_user']
X = df.drop('is_new_user', axis=1)

# [중요] 결측치 처리: 숫자형은 중앙값, 문자열형은 최빈값으로 채움
# [重要] 缺失值处理：数值型用中位数填充，字符型用众数填充
# [Important] Missing value handling: fill numeric with median, string with mode
for col in X.columns:
    if X[col].dtype in ['int64', 'float64']:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode()[0])

# [중요] 범주형 변수 원-핫 인코딩 (KNN, SVM은 숫자만 인식하므로 필수)
# [重要] 分类变量独热编码（KNN、SVM仅识别数字，因此必需）
# [Important] One-hot encoding for categorical variables (essential since KNN and SVM only recognize numbers)
X = pd.get_dummies(X, drop_first=True)

# Target 변수가 문자열(예: 'Yes'/'No')일 경우 1/0으로 변환
# 若目标变量为字符串（如'Yes'/'No'），则转换为1/0
# [수정 2026-11] 원본의 'No': 1 → 'No': 0 / [修正] 'No': 1 → 'No': 0
# If Target variable is a string (e.g., 'Yes'/'No'), convert to 1/0
if y.dtype == 'object' or y.dtype == 'bool':
    y = y.replace({'True': 1, 'False': 0, 'Yes': 1, 'No': 0, 'Y': 1, 'N': 0})
y = y.astype(int)

# ==========================================
# 3. 데이터 분할 (훈련 80%, 테스트 20%)
# 3. 数据分割（训练集80%，测试集20%）
# 3. Data Split (Training 80%, Test 20%)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ 데이터 분할 완료 (Train: {X_train.shape[0]}개, Test: {X_test.shape[0]}개)")

# ==========================================
# 4. 데이터 스케일링 (표준화)
# 4. 数据缩放（标准化）
# 4. Data Scaling (Standardization)
# ==========================================
# KNN과 SVM은 거리 기반 알고리즘이므로 스케일링이 성능에 절대적 영향을 미칩니다.
# KNN和SVM是基于距离的算法，因此缩放对性能有绝对影响。
# KNN and SVM are distance-based algorithms, so scaling has an absolute impact on performance.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 5. 모델 학습 (KNN, SVM)
# 5. 模型训练（KNN，SVM）
# 5. Model Training (KNN, SVM)
# ==========================================
# KNN 모델 (이웃 수 k=5)
# KNN模型（邻居数k=5）
# KNN Model (number of neighbors k=5)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# SVM 모델 (ROC 곡선 작성을 위해 probability=True 설정)
# SVM模型（为绘制ROC曲线设置probability=True）
# SVM Model (set probability=True for ROC curve plotting)
svm = SVC(probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
print("✅ 모델 학습 완료!")

# ==========================================
# 6. 예측 및 평가 지표 출력
# 6. 预测及评估指标输出
# 6. Prediction and Evaluation Metrics Output
# ==========================================
y_pred_knn = knn.predict(X_test_scaled)
y_pred_svm = svm.predict(X_test_scaled)

print("\n" + "="*40)
print("📊 [모델 정확도 (Accuracy) 비교]")
print(f"KNN 정확도: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"SVM 정확도: {accuracy_score(y_test, y_pred_svm):.4f}")
print("="*40)

print("\n📝 [KNN 분류 리포트]")
print(classification_report(y_test, y_pred_knn, target_names=['기존 유저(0)', '신규 유저(1)']))

# ==========================================
# 7. 결과 시각화 (그래프)
# 7. 结果可视化（图表）
# 7. Result Visualization (Graphs)
# ==========================================
plt.figure(figsize=(15, 12))

# 1) 혼동 행렬 (Confusion Matrix) 비교
# 1) 混淆矩阵（Confusion Matrix）比较
# 1) Confusion Matrix Comparison
plt.subplot(2, 2, 1)
cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['기존(0)', '신규(1)'], yticklabels=['기존(0)', '신규(1)'])
plt.title('KNN 혼동 행렬 (Confusion Matrix)')
plt.ylabel('실제 값 (Actual)')
plt.xlabel('예측 값 (Predicted)')

plt.subplot(2, 2, 2)
cm_svm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['기존(0)', '신규(1)'], yticklabels=['기존(0)', '신규(1)'])
plt.title('SVM 혼동 행렬 (Confusion Matrix)')
plt.ylabel('실제 값 (Actual)')
plt.xlabel('예측 값 (Predicted)')

# 2) ROC Curve 비교
# 2) ROC曲线比较
# 2) ROC Curve Comparison
plt.subplot(2, 1, 2)
# KNN ROC
# KNN ROC
# KNN ROC
fpr_knn, tpr_knn, _ = roc_curve(y_test, knn.predict_proba(X_test_scaled)[:, 1])
roc_auc_knn = auc(fpr_knn, tpr_knn)
plt.plot(fpr_knn, tpr_knn, color='blue', linestyle='-', linewidth=2, label=f'KNN (AUC = {roc_auc_knn:.3f})')

# SVM ROC
# SVM ROC
# SVM ROC
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm.predict_proba(X_test_scaled)[:, 1])
roc_auc_svm = auc(fpr_svm, tpr_svm)
plt.plot(fpr_svm, tpr_svm, color='red', linestyle='-', linewidth=2, label=f'SVM (AUC = {roc_auc_svm:.3f})')

# 대각선 기준선
# 对角线基准线
# Diagonal baseline
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('위양성률 (False Positive Rate)')
plt.ylabel('진양성률 (True Positive Rate)')
plt.title('ROC Curve 비교 (곡선 아래 면적 AUC가 1에 가까울수록 좋은 모델)')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

# 3) 정확도 막대 그래프
# 3) 准确率柱状图
# 3) Accuracy Bar Chart
models = ['KNN', 'SVM']
accuracies = [accuracy_score(y_test, y_pred_knn), accuracy_score(y_test, y_pred_svm)]

plt.figure(figsize=(6, 4))
sns.barplot(x=models, y=accuracies, palette='Set2')
plt.title('모델별 정확도 (Accuracy) 비교')
plt.ylabel('정확도')
plt.ylim(0, 1)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')
plt.show()